## Topic: Introduction of SQLite Database

In [ ]:
# from langgraph.checkpoint.sqlite import SqliteSaver

c:\Users\kz\anaconda3\envs\GenAI\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.7.0) or chardet (7.6.0)/charset_normalizer (3.5.1) doesn't match a supported version!
  warnings.warn(
c:\Users\kz\anaconda3\envs\GenAI\Lib\site-packages\langgraph\checkpoint\base\__init__.py:18: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


### 1. Introduction of Sqlite Database

- Definition:
    - SQLite is a self-contained, serverless, zero-configuration, transactional SQL database engine. Unlike traditional databases that run as a separate server process, SQLite is a library that reads and writes directly to a single file on disk.


In [ ]:
""" 
Traditional Database (e.g., PostgreSQL, MySQL):

  ┌──────────────┐         ┌──────────────────┐
  │  Your App    │──TCP──▶ │  Database SERVER │
  │  (Client)    │  IP     │  (Separate       │
  │              │◀──TCP── │   Process)       │
  └──────────────┘         │    │             │
                           │    ▼             │
  Requires:                │  ┌───────────┐   │
  • Server installation    │  │ Data Files│   │
  • Port configuration     │  │ (on disk) │   │
  • User management        │  └───────────┘   │
  • Network connection     └──────────────────┘
  • Running daemon

SQLite Database:

  ┌──────────────────────────────────┐
  │  Your App                        │
  │    │                             │
  │    ▼                             │
  │  ┌──────────┐   ┌─────────────┐  │
  │  │ SQLite   │──▶│ single_file │  │
  │  │ Library  │◀──│   .db       │  │
  │  └──────────┘   └─────────────┘  │
  │  (linked directly                │
  │   into your app)                 │
  └──────────────────────────────────┘

  Requires:
  • Nothing! Just a file.
  • No server. No port. No daemon.
  • The database IS the file.

- Main idea: SQLite is not a client-server database. It is an embedded database. The entire database — tables, indexes, data — lives in one ordinary file on your computer (e.g., my_database.db). Your application reads and writes to this file directly through the SQLite library.
"""

### 2. Why SQLite? (The World's Most Deployed Database)

In [ ]:
""" 
┌─────────────────────────────────────────────────────────────┐
│           SQLITE BY THE NUMBERS                             │
│                                                             │
│  📱 Every smartphone has SQLite                              │
│     • iOS: Core Data runs on SQLite                         │
│     • Android: Built-in SQLite for app storage              │
│     → ~4 billion phones × ~10 databases each                │
│                                                             │
│  🌐 Every web browser has SQLite                            │
│     • Chrome, Firefox, Safari, Edge                         │
│     • Used for bookmarks, history, cookies, cache           │
│                                                             │
│  💻 Every major OS ships SQLite                             │
│     • macOS, Windows 10+, Linux distributions               │
│                                                             │
│  📊 Estimated deployments: 1 TRILLION+                      │
│     More than all other database engines COMBINED.          │
│                                                             │
│  📦 Size: ~600 KB (the entire engine!)                      │
│     Smaller than most JPEG images.                          │
│                                                             │
│  🔋 Zero configuration                                      │
│     No install. No setup. No admin. Just use it.            │
│                                                             │
│  🔒 ACID compliant                                          │
│     Full transaction support. Data is safe.                 │
└─────────────────────────────────────────────────────────────┘

"""

In [ ]:
""" 
- SQLite in the LangGraph Context (Checkpointers):
==================================================
How LangGraph Uses SQLite:

  ┌─────────────────────────────────────────────────────┐
  │  Your LangGraph Application                         │
  │                                                     │
  │  graph = builder.compile(                           │
  │      checkpointer=SqliteSaver(conn)                 │
  │  )                                                  │
  │         │                                           │
  │         ▼                                           │
  │  After each node execution:                         │
  │  ┌─────────────────────────────────────────────┐    │
  │  │ SqliteSaver writes to the .db file:         │    │
  │  │                                             │    │
  │  │  Table: checkpoints                         │    │
  │  │  ┌──────────┬──────────┬─────────────────┐  │    │
  │  │  │ thread_id│ check_id │ state (BLOB)    │  │    │
  │  │  ├──────────┼──────────┼─────────────────┤  │    │
  │  │  │ thread_1 │ cp_001   │ {messages:[...]}│  │    │
  │  │  │ thread_1 │ cp_002   │ {messages:[...]}│  │    │
  │  │  │ thread_2 │ cp_003   │ {messages:[...]}│  │    │
  │  │  └──────────┴──────────┴─────────────────┘  │    │
  │  │                                             │    │
  │  │  Table: checkpoint_writes                   │    │
  │  │  Table: checkpoint_blobs                    │    │
  │  └─────────────────────┬───────────────────────┘    │
  │                        │                            │
  └────────────────────────┼────────────────────────────┘
                           ▼
                  ┌─────────────────┐
                  │ checkpoints.db  │  ← Single file on disk
                  └─────────────────┘
                     Survives restarts!

"""

### Key Takeaways

In [ ]:
""" 
┌─────────────────────────────────────────────────────────────┐
│           SQLITE — KEY TAKEAWAYS                            │
│                                                             │
│  1. ONE FILE = ONE DATABASE                                 │
│     The entire database lives in a single .db file.         │
│     Copy the file = backup the database.                    │
│     Delete the file = delete the database.                  │
│                                                             │
│  2. ZERO CONFIGURATION                                      │
│     No server. No port. No installation. No admin.          │
│     import sqlite3 → conn = sqlite3.connect("app.db")       │
│     That's it. You have a database.                         │
│                                                             │
│  3. BUILT INTO PYTHON                                       │
│     The sqlite3 module is part of the Python standard       │
│     library. No pip install needed.                         │
│                                                             │
│  4. FULL SQL SUPPORT                                        │
│     SELECT, INSERT, UPDATE, DELETE, JOIN, GROUP BY,         │
│     subqueries, CTEs, window functions — all supported.     │
│                                                             │
│  5. ACID COMPLIANT                                          │
│     Transactions are safe. Data won't corrupt on crash.     │
│                                                             │
│  6. SINGLE WRITER LIMITATION                                │
│     Only one write at a time. Fine for most use cases,      │
│     but a bottleneck for high-concurrency apps.             │
│                                                             │
│  7. ALWAYS USE PARAMETERIZED QUERIES                        │
│     cursor.execute("... WHERE id = ?", (id,))  ✅           │
│     cursor.execute(f"... WHERE id = {id}")     ❌ DANGER    │
│                                                             │
│  8. PERFECT FOR LANGGRAPH CHECKPOINTING                     │
│     SqliteSaver gives you durable persistence with          │
│     zero infrastructure. Ideal for single-server agents.    │
│     Upgrade to PostgresSaver when you need scale.           │
│                                                             │
│  9. THE WORLD'S MOST DEPLOYED DATABASE                      │
│     In your phone, browser, OS, and now your AI agents.     │
│     ~1 trillion deployments and counting.                   │
│                                                             │
│  10. START WITH SQLITE, UPGRADE WHEN NEEDED                 │
│      SQLite → PostgreSQL is a common growth path.           │
│      LangGraph makes this easy: just swap the checkpointer. │
└─────────────────────────────────────────────────────────────┘


"""